# **MapReduce_Ordenamiento**

**Crear "chunks"**

In [7]:
import os

os.makedirs("chunks", exist_ok=True)

datos_por_chunk = {
    "chunk_1.txt": [8, 3, 10, 1, 7],
    "chunk_2.txt": [5, 20, 11, 4, 9],
    "chunk_3.txt": [15, 2, 18, 6, 14],
    "chunk_4.txt": [13, 12, 16, 0, 19]
}

for nombre, datos in datos_por_chunk.items():
    with open(os.path.join("chunks", nombre), "w", encoding="utf-8") as f:
        for x in datos:
            f.write(str(x) + "\n")

print("Chunks creados.")

Chunks creados.


**Fase Map**

In [8]:
def map_sort(nombre_archivo, num_linea, contenido_linea):
    valor = int(contenido_linea.strip())
    return (valor, 1)

**Leer chunks y aplicar map**

In [9]:
mapped_por_chunk = {}

for nombre_archivo in sorted(os.listdir("chunks")):
    ruta = os.path.join("chunks", nombre_archivo)
    pares = []
    
    with open(ruta, "r", encoding="utf-8") as f:
        for num_linea, linea in enumerate(f, start=1):
            pares.append(map_sort(nombre_archivo, num_linea, linea))
    
    mapped_por_chunk[nombre_archivo] = pares

for chunk, pares in mapped_por_chunk.items():
    print(f"{chunk} -> {pares}")

chunk_1.txt -> [(8, 1), (3, 1), (10, 1), (1, 1), (7, 1)]
chunk_2.txt -> [(5, 1), (20, 1), (11, 1), (4, 1), (9, 1)]
chunk_3.txt -> [(15, 1), (2, 1), (18, 1), (6, 1), (14, 1)]
chunk_4.txt -> [(13, 1), (12, 1), (16, 1), (0, 1), (19, 1)]


**Particionador por rango**

**Nota:** no se utilizo partición por hash, ya que no mantiene el orden necesariamente al final al concatenar. Por lo que se Particiono por un rango $clave<7$, $clave<14$.

In [10]:
def range_partitioner(clave):
    if clave < 7:
        return 0
    elif clave < 14:
        return 1
    else:
        return 2

**Shuffle hacia reducers**

In [11]:
from collections import defaultdict

reducers_input = defaultdict(list)

for chunk, pares in mapped_por_chunk.items():
    for clave, valor in pares:
        reducer_id = range_partitioner(clave)
        reducers_input[reducer_id].append((clave, valor))

for reducer_id in sorted(reducers_input):
    print(f"\nReducer {reducer_id} recibe:")
    print(reducers_input[reducer_id])


Reducer 0 recibe:
[(3, 1), (1, 1), (5, 1), (4, 1), (2, 1), (6, 1), (0, 1)]

Reducer 1 recibe:
[(8, 1), (10, 1), (7, 1), (11, 1), (9, 1), (13, 1), (12, 1)]

Reducer 2 recibe:
[(20, 1), (15, 1), (18, 1), (14, 1), (16, 1), (19, 1)]


 **Agrupar y ordenar dentro de cada reducer**

In [13]:
grouped_reducers = {}

for reducer_id, pares in reducers_input.items():
    grouped = defaultdict(list)
    for clave, valor in pares:
        grouped[clave].append(valor)
    
    grouped_ordenado = dict(sorted(grouped.items()))
    grouped_reducers[reducer_id] = grouped_ordenado

for reducer_id in sorted(grouped_reducers):
    print(f"\nReducer {reducer_id} agrupado y ordenado:")
    for k, v in grouped_reducers[reducer_id].items():
        print(k, v)


Reducer 0 agrupado y ordenado:
0 [1]
1 [1]
2 [1]
3 [1]
4 [1]
5 [1]
6 [1]

Reducer 1 agrupado y ordenado:
7 [1]
8 [1]
9 [1]
10 [1]
11 [1]
12 [1]
13 [1]

Reducer 2 agrupado y ordenado:
14 [1]
15 [1]
16 [1]
18 [1]
19 [1]
20 [1]


**Reduce**

In [14]:
def reduce_sort(clave, lista_valores):
    salida = []
    for _ in lista_valores:
        salida.append(clave)
    return salida

**Salida por Reducer**

In [15]:
salidas_reducers = {}

for reducer_id, grouped in grouped_reducers.items():
    salida = []
    for clave, lista_valores in grouped.items():
        salida.extend(reduce_sort(clave, lista_valores))
    salidas_reducers[reducer_id] = salida

for reducer_id in sorted(salidas_reducers):
    print(f"Salida Reducer {reducer_id}: {salidas_reducers[reducer_id]}")

Salida Reducer 0: [0, 1, 2, 3, 4, 5, 6]
Salida Reducer 1: [7, 8, 9, 10, 11, 12, 13]
Salida Reducer 2: [14, 15, 16, 18, 19, 20]


**Concatenando las salidas**

In [16]:
resultado_final = []

for reducer_id in sorted(salidas_reducers):
    resultado_final.extend(salidas_reducers[reducer_id])

print("\nResultado final globalmente ordenado:")
print(resultado_final)


Resultado final globalmente ordenado:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20]
